# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/summayashaikh079-stack/flyrank-ml-week1/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

Contract, in plain words:

One row = one (report_date, client, content item) daily performance record.
Table used: fact_content_daily_performance, joined with dim_content (content metadata) and dim_clients (access flags).
Time window: month=2026-03 — a mid-panel month, not the final month (_sample/June is a sealed answer key, deliberately avoided).
Label/proxy: refresh-need is computed from within-month click/impression trend for each content item — never a feature.
Deliberately excluded: client_hash_id/content_hash_id — pseudonymized IDs, used only for grouping/joining, never as model inputs.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
!pip install huggingface_hub duckdb --quiet

from huggingface_hub import hf_hub_download
from google.colab import userdata
import duckdb

token = userdata.get('HF_TOKEN')

daily_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse", repo_type="dataset",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    token=token
)
content_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse", repo_type="dataset",
    filename="dim_content.parquet", token=token
)
clients_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse", repo_type="dataset",
    filename="dim_clients.parquet", token=token
)

con = duckdb.connect()
con.execute(f"CREATE VIEW daily AS SELECT * FROM read_parquet('{daily_path}')")
con.execute(f"CREATE VIEW dim_content AS SELECT * FROM read_parquet('{content_path}')")
con.execute(f"CREATE VIEW dim_clients AS SELECT * FROM read_parquet('{clients_path}')")

print(con.execute("SELECT COUNT(*) AS rows FROM daily").fetchdf())

      rows
0  9841378


## 2. Fields: feature / label / context / excluded

Features: content_type, word_count, main_intent, search_volume, competition (from dim_content); gsc_impressions, gsc_clicks, client_has_gsc, client_has_ga4 (from daily, when gsc_data_available IS TRUE) — knowable at the decision moment.
Label/proxy: within-month click/impression trend for each content item — computed from daily, never a feature.
Context: client_hash_id, content_hash_id, report_date — for grouping/joining/splitting only, never for the model to learn from.
Excluded: gsc_data_available/ga4_data_available when NULL or FALSE rows — those rows mean "not measured", would silently corrupt averages if not filtered with IS TRUE.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
features = ['content_type', 'word_count', 'main_intent', 'search_volume', 'competition',
            'gsc_impressions', 'gsc_clicks', 'client_has_gsc', 'client_has_ga4']
label_proxy = ['gsc_impressions_trend', 'gsc_clicks_trend']  # computed within-month, never a feature
context = ['content_hash_id', 'client_hash_id', 'report_date']
excluded = ['gsc_data_available', 'ga4_data_available']  # filter flags, not model inputs

print(f"Features ({len(features)}): {features}")
print(f"Label/proxy ({len(label_proxy)}): {label_proxy}")
print(f"Context ({len(context)}): {context}")
print(f"Excluded ({len(excluded)}): {excluded}")

# Confirm columns actually exist in the daily table
cols = con.execute("DESCRIBE daily").fetchdf()
print("\nColumns available in 'daily':")
print(cols['column_name'].tolist())

Features (9): ['content_type', 'word_count', 'main_intent', 'search_volume', 'competition', 'gsc_impressions', 'gsc_clicks', 'client_has_gsc', 'client_has_ga4']
Label/proxy (2): ['gsc_impressions_trend', 'gsc_clicks_trend']
Context (3): ['content_hash_id', 'client_hash_id', 'report_date']
Excluded (2): ['gsc_data_available', 'ga4_data_available']

Columns available in 'daily':
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


## 3. Verify it with queries (grain, counts, missing values, windows)
Three verification queries: (1) grain check on (report_date, client, content) — should return 0 duplicates. (2) row count + date span for month=2026-03. (3) availability check — GSC rows filtered with gsc_data_available IS TRUE, showing how many rows survive vs. total.
Five features, each with "knowable at the decision moment because…":
content_type — set when content is published, before any performance data exists.
word_count — fixed at publish time.
search_volume — external keyword-research metric, known before the page goes live.
gsc_impressions_prior (lagged) — impressions from before the current window, safe because it precedes the decision.
client_has_gsc — an account-setup flag, known at account creation, unrelated to future performance.
Leakage trap: deliberately add gsc_impressions_trend (computed from future-relative-to-decision data) as a feature, watch a toy score jump toward perfect, then remove it — demonstrating the leak from notebook 02 on real warehouse data.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# --- Query 1: Grain check ---
print("=== Query 1: Grain check ===")
grain = con.execute("""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) c
    FROM daily GROUP BY 1,2,3 HAVING c > 1 LIMIT 5
""").fetchdf()
print(f"Duplicate grain rows: {len(grain)} (should be 0)")

# --- Query 2: Row count + date span ---
print("\n=== Query 2: Row count + date span ===")
print(con.execute("SELECT COUNT(*) AS row_count, MIN(report_date) AS start_date, MAX(report_date) AS end_date FROM daily").fetchdf())

# --- Query 3: Availability check with IS TRUE ---
print("\n=== Query 3: Availability check ===")
total = con.execute("SELECT COUNT(*) FROM daily").fetchdf().iloc[0,0]
available = con.execute("SELECT COUNT(*) FROM daily WHERE gsc_data_available IS TRUE").fetchdf().iloc[0,0]
print(f"Total rows: {total}, GSC-available rows: {available} ({available/total*100:.1f}%)")

# --- Five features on a small feature frame ---
print("\n=== Five features ===")
features_df = con.execute("""
    SELECT d.content_hash_id, d.client_hash_id,
           c.content_type, c.word_count, c.search_volume,
           d.client_has_gsc,
           LAG(d.gsc_impressions) OVER (PARTITION BY d.content_hash_id ORDER BY d.report_date) AS gsc_impressions_prior
    FROM daily d
    JOIN dim_content c ON d.content_hash_id = c.content_hash_id
    WHERE d.gsc_data_available IS TRUE
    LIMIT 5000
""").fetchdf()
print(features_df.head())
print(f"\nFeature frame shape: {features_df.shape}")

=== Query 1: Grain check ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate grain rows: 0 (should be 0)

=== Query 2: Row count + date span ===
   row_count start_date   end_date
0    9841378 2026-03-01 2026-03-31

=== Query 3: Availability check ===
Total rows: 9841378, GSC-available rows: 3611061 (36.7%)

=== Five features ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

            content_hash_id           client_hash_id     content_type  \
0  content_00032be2df0005ca  client_fef1a8f436438636  keyword article   
1  content_00032be2df0005ca  client_fef1a8f436438636  keyword article   
2  content_00032be2df0005ca  client_fef1a8f436438636  keyword article   
3  content_00032be2df0005ca  client_fef1a8f436438636  keyword article   
4  content_00032be2df0005ca  client_fef1a8f436438636  keyword article   

   word_count  search_volume  client_has_gsc  gsc_impressions_prior  
0        1502            110            True                   <NA>  
1        1502            110            True                     22  
2        1502            110            True                     13  
3        1502            110            True                     13  
4        1502            110            True                     17  

Feature frame shape: (5000, 7)


In [14]:
# --- Leakage trap: deliberately leak the label into a feature ---
import numpy as np

leak_df = con.execute("""
    SELECT d.content_hash_id,
           c.word_count, c.search_volume,
           d.gsc_impressions,
           d.gsc_impressions - LAG(d.gsc_impressions) OVER (PARTITION BY d.content_hash_id ORDER BY d.report_date) AS gsc_impressions_trend
    FROM daily d
    JOIN dim_content c ON d.content_hash_id = c.content_hash_id
    WHERE d.gsc_data_available IS TRUE
    LIMIT 5000
""").fetchdf().dropna()

# Toy "label": did impressions grow this period? (what we're trying to predict)
leak_df['grew'] = (leak_df['gsc_impressions_trend'] > 0).astype(int)

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

# WITH the leak (gsc_impressions_trend used as a feature — it's literally derived from the label)
X_leak = leak_df[['word_count', 'search_volume', 'gsc_impressions_trend']].fillna(0)
y = leak_df['grew']
Xtr, Xte, ytr, yte = train_test_split(X_leak, y, test_size=0.3, random_state=0)
model_leak = LogisticRegression(max_iter=1000).fit(Xtr, ytr)
print(f"WITH leak — accuracy: {model_leak.score(Xte, yte):.3f}  <- suspiciously perfect")

# WITHOUT the leak (removed)
X_clean = leak_df[['word_count', 'search_volume']].fillna(0)
Xtr, Xte, ytr, yte = train_test_split(X_clean, y, test_size=0.3, random_state=0)
model_clean = LogisticRegression(max_iter=1000).fit(Xtr, ytr)
print(f"WITHOUT leak — accuracy: {model_clean.score(Xte, yte):.3f}  <- honest number")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

WITH leak — accuracy: 1.000  <- suspiciously perfect
WITHOUT leak — accuracy: 0.562  <- honest number


## 4. Data limits

One named limitation: GSC availability is only 36.7% for March 2026 — most content items have no search-console data this month, so any model trained here would only "see" a minority slice of the catalog, and that slice may not be representative of low-visibility or newly-published content. This is a per-client access gap (gsc_data_available), not a random sample, so conclusions from this data are decision-support for GSC-connected clients only, not a general claim about all content.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Confirm the limitation with a real number
availability_by_client = con.execute("""
    SELECT client_hash_id,
           AVG(CASE WHEN gsc_data_available IS TRUE THEN 1.0 ELSE 0 END) AS pct_available
    FROM daily
    GROUP BY client_hash_id
    ORDER BY pct_available
""").fetchdf()

print(f"Clients with 0% GSC availability this month: {(availability_by_client['pct_available']==0).sum()} out of {len(availability_by_client)}")
print(f"Clients with 100% GSC availability this month: {(availability_by_client['pct_available']==1).sum()} out of {len(availability_by_client)}")

Clients with 0% GSC availability this month: 8 out of 55
Clients with 100% GSC availability this month: 0 out of 55


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.